### 📈 Tool 2: Stock performance data

You'll create a tool to load stock performance data from a local CSV file, filter it depending on the user's query, then return the results.

This tool will use stock _ticker symbols_, which are unique company IDs, to return the relevant information for that company. Here are the companies and symbols that have stock data available in your environment:

| Company Name | Ticker Symbol |
|--------------|---------------|
| Apple        | AAPL          |
| Microsoft    | MSFT          |
| Amazon       | AMZN          |
| Meta         | META          |
| Netflix      | NFLX          |
| Tesla        | TSLA          |

Note that this data isn't up-to-date, but you could adapt this to pull real-time data from an API like Yahoo Finance, rather than the CSVs. Try this out after completing the course!

In [38]:
%pip install langchain-core==0.3.59 
%pip install pydantic==2.11.9 
%pip install pandas
%pip install tabulate


Note: you may need to restart the kernel to use updated packages.
  Using cached pydantic-2.11.9-py3-none-any.whl.metadata (68 kB)
  Using cached pydantic_core-2.33.2.tar.gz (435 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Checking for Rust toolchain....
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp314-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\IT Intern\AppData\Local\puccinialin\puccinialin\Cache
      Rustup already downloaded
      Installing rust to C:\Users\IT Intern\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: It looks like you have an existing rustup settings file at:
      warn: C:\Users\IT Intern\.rustup\settings.toml
      warn: Rustup will install the default toolchain as specified in the settings file,
      warn: instead of the one inferred from the default host triple.
      info: profile set to 'minimal'
      info: default host triple is x86_64-pc-windows-msvc
      warn: Updating existing toolchain, profile 

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [39]:
import os
from typing import Annotated
import pandas as pd
from langchain_core.tools import tool

@tool
def stock_data_tool(
    company_ticker: Annotated[str, "The ticker symbol of the company to retrieve their stock performance data."], 
    num_days: Annotated[int, "The number of days of stock data required to respond to the user query."]
) -> str:
    """
    Use this to look-up stock performance data for companies to retrieve a table from a CSV. You may need to convert company names into ticker symbols to call this function, e.g, Apple Inc. -> AAPL, and you may need to convert weeks, months, and years, into days.
    """
    
    # Load the CSV for the company requested
    file_path = f"data/{company_ticker}.csv"

    if os.path.exists(file_path) is False:
        return f"Sorry, but data for company {company_ticker} is not available. Please try Apple, Amazon, Meta, Microsoft, Netflix, or Tesla."
    
    stock_df = pd.read_csv(file_path, index_col='Date', parse_dates=True)

    # Ensure the index is in date format
    stock_df.index = stock_df.index.date
    
    # Maximum num_days supported by the dataset
    max_num_days = (stock_df.index.max() - stock_df.index.min()).days
    
    if num_days > max_num_days:
        return "Sorry, but this time period exceeds the data available. Please reduce it to continue."
    
    # Get the most recent date in the DataFrame
    final_date = stock_df.index.max()

    # Filter the DataFrame to get the last num_days of stock data
    filtered_df = stock_df[stock_df.index > (final_date - pd.Timedelta(days=num_days))]

    return f"Successfully executed the stock performance data retrieval tool to retrieve the last *{num_days} days* of data for company **{company_ticker}**:\n\n{filtered_df.to_markdown()}"

In [43]:
retrieved_data = stock_data_tool.invoke({"company_ticker": "META", "num_days": 30})
print(retrieved_data)

Successfully executed the stock performance data retrieval tool to retrieve the last *30 days* of data for company **META**:

|            | Close/Last   |   Volume | Open     | High      | Low       |
|:-----------|:-------------|---------:|:---------|:----------|:----------|
| 2025-05-28 | $643.58      |  9042874 | $642.60  | $650.88   | $642.5472 |
| 2025-05-27 | $642.32      |  9508367 | $635.41  | $643.08   | $632.75   |
| 2025-05-23 | $627.06      |  8454067 | $624.00  | $632.445  | $622.65   |
| 2025-05-22 | $636.57      |  8228443 | $634.05  | $643.25   | $630.71   |
| 2025-05-21 | $635.50      | 11464570 | $631.79  | $646.61   | $630.17   |
| 2025-05-20 | $637.10      |  6743473 | $636.01  | $639.3536 | $632.26   |
| 2025-05-19 | $640.43      |  9592374 | $628.25  | $643.00   | $627.80   |
| 2025-05-16 | $640.34      | 18518970 | $637.955 | $640.44   | $626.15   |
| 2025-05-15 | $643.88      | 14341840 | $654.275 | $657.31   | $638.58   |
| 2025-05-14 | $659.36      | 12348180

In [44]:
from IPython.display import display, Markdown
display(Markdown(retrieved_data))

Successfully executed the stock performance data retrieval tool to retrieve the last *30 days* of data for company **META**:

|            | Close/Last   |   Volume | Open     | High      | Low       |
|:-----------|:-------------|---------:|:---------|:----------|:----------|
| 2025-05-28 | $643.58      |  9042874 | $642.60  | $650.88   | $642.5472 |
| 2025-05-27 | $642.32      |  9508367 | $635.41  | $643.08   | $632.75   |
| 2025-05-23 | $627.06      |  8454067 | $624.00  | $632.445  | $622.65   |
| 2025-05-22 | $636.57      |  8228443 | $634.05  | $643.25   | $630.71   |
| 2025-05-21 | $635.50      | 11464570 | $631.79  | $646.61   | $630.17   |
| 2025-05-20 | $637.10      |  6743473 | $636.01  | $639.3536 | $632.26   |
| 2025-05-19 | $640.43      |  9592374 | $628.25  | $643.00   | $627.80   |
| 2025-05-16 | $640.34      | 18518970 | $637.955 | $640.44   | $626.15   |
| 2025-05-15 | $643.88      | 14341840 | $654.275 | $657.31   | $638.58   |
| 2025-05-14 | $659.36      | 12348180 | $661.21  | $662.67   | $654.31   |
| 2025-05-13 | $656.03      | 18570820 | $645.545 | $660.92   | $642.83   |
| 2025-05-12 | $639.43      | 21965090 | $630.92  | $640.3896 | $621.03   |
| 2025-05-09 | $592.49      | 10427290 | $603.72  | $606.97   | $591.7062 |
| 2025-05-08 | $598.01      | 14622810 | $606.285 | $611.2959 | $596.62   |
| 2025-05-07 | $596.81      | 13160980 | $590.36  | $603.075  | $586.67   |
| 2025-05-06 | $587.31      | 10600650 | $592.525 | $596.03   | $586.58   |
| 2025-05-05 | $599.27      | 13887720 | $591.22  | $603.21   | $588.05   |
| 2025-05-02 | $597.02      | 24739260 | $583.455 | $604.34   | $578.33   |
| 2025-05-01 | $572.21      | 31159030 | $592.075 | $592.95   | $570.50   |
| 2025-04-30 | $549.00      | 29243970 | $538.40  | $549.10   | $529.50   |
| 2025-04-29 | $554.44      | 11835040 | $546.00  | $556.5699 | $544.12   |